# Fase 5a — Creación de los índices de Elasticsearch

Conecta con una instancia local de Elasticsearch y crea los **cuatro índices** que almacenarán los chunks generados por las cuatro estrategias de chunking de la Fase 3b: `indice_estrategia_a_fixed`, `indice_estrategia_b_markdown`, `indice_estrategia_c_semantica` e `indice_estrategia_d_jerarquico`.

Se combina un campo de texto para la búsqueda léxica (BM25) con **tres vectores densos** (`vector_generalista`, `vector_medico`, `vector_experto`; 768 dimensiones, similitud coseno) que permiten comparar, sobre el mismo contenido, distintos modelos de embeddings (Fase 5c). Los índices B y D añaden además los campos de sección/subsección Markdown, y el D incorpora `parent_id`/`doc_type` para conservar la relación padre-hijo de la estrategia jerárquica.

La última celda lista los índices resultantes en el clúster a modo de verificación.

In [3]:
!pip install elasticsearch

In [4]:
from elasticsearch import Elasticsearch

# 1. Conexión
es = Elasticsearch("http://localhost:9200",
                   request_timeout=60,      # Esperamos hasta 60 segundos por respuesta
                   max_retries=3,           # Si falla, lo intenta 3 veces más
                   retry_on_timeout=True    # Permite reintentar si da timeout
                   )
print("Conectado a Elasticsearch:", es.ping())

# 2. Variables para las dimensiones (¡Las cambiaremos luego!)
DIM_GENERALISTA = 768  # e5-base
DIM_MEDICO = 768       # bsc-bio-es
DIM_EXPERTO = 768

# 3. Función para generar el mapping base
def generar_mapping(extra_metadata_fields=None):
    """
    Construye el mapping (esquema) de un índice de Elasticsearch para
    búsqueda híbrida de chunks RAG: un campo de texto para la búsqueda
    léxica BM25, tres vectores densos (generalista, médico y experto, 768
    dimensiones cada uno, similitud coseno) para comparar distintos modelos
    de embeddings sobre el mismo contenido, y los metadatos base comunes a
    todos los chunks, ampliables con campos extra propios de cada estrategia
    de chunking (p. ej. secciones Markdown o relación padre-hijo).
    """
    # Metadatos base que tienen TODOS los chunks
    metadata_properties = {
        "source": {"type": "keyword"},
        "domain": {"type": "keyword"},
        "type": {"type": "keyword"},
        "titulo": {"type": "keyword"},
        "autores": {"type": "text"}, # 'text' por si quieres buscar por un autor suelto
        "fecha_publicacion": {"type": "keyword"},
        "tema": {"type": "keyword"},
        "chunk_id": {"type": "keyword"},
        "chunk_index": {"type": "integer"},
        "estrategia": {"type": "keyword"}
    }

    # Si hay campos extra (como en la estrategia B), los añadimos
    if extra_metadata_fields:
        metadata_properties.update(extra_metadata_fields)

    return {
        "mappings": {
            "properties": {
                "chunk_id": {"type": "keyword"},
                "text": {"type": "text"}, # Aquí actúa el BM25

                # Vectores
                "vector_generalista": {
                    "type": "dense_vector",
                    "dims": DIM_GENERALISTA,
                    "index": True,
                    "similarity": "cosine"
                },
                "vector_medico": {
                    "type": "dense_vector",
                    "dims": DIM_MEDICO,
                    "index": True,
                    "similarity": "cosine"
                },
                "vector_experto": {
                    "type": "dense_vector",
                    "dims": DIM_EXPERTO,
                    "index": True,
                    "similarity": "cosine"
                },

                # Metadatos
                "metadata": {
                    "properties": metadata_properties
                }
            }
        }
    }

# 4. Definición de los 3 índices y sus mappings específicos
indices_a_crear = {
    "indice_estrategia_a_fixed": generar_mapping(),

    # A la estrategia B le añadimos sus campos exclusivos (opcionales)
    "indice_estrategia_b_markdown": generar_mapping({
        "seccion": {"type": "keyword"},
        "subseccion": {"type": "keyword"}
    }),

    "indice_estrategia_c_semantica": generar_mapping(),

    "indice_estrategia_d_jerarquico": generar_mapping({
        "seccion": {"type": "keyword"},
        "subseccion": {"type": "keyword"},
        "parent_id": {"type": "keyword"}, # Clave para enlazar al padre
        "doc_type": {"type": "keyword"}   # Clave para saber si es "padre" o "hijo"
    })
}

# 5. Ejecutar la creación
for index_name, mapping in indices_a_crear.items():
    if not es.indices.exists(index=index_name):
        es.indices.create(index=index_name, body=mapping)
        print(f"Índice creado: {index_name}")
    else:
        print(f"El índice ya existía: {index_name}")

Conectado a Elasticsearch: True
Índice creado: indice_estrategia_a_fixed
Índice creado: indice_estrategia_b_markdown
Índice creado: indice_estrategia_c_semantica
Índice creado: indice_estrategia_d_jerarquico


In [5]:
# Muestra una tabla en texto con todos los índices del clúster
print(es.cat.indices(v=True))

health status index                          uuid                   pri rep docs.count docs.deleted store.size pri.store.size dataset.size
yellow open   indice_estrategia_a_fixed      ogWoF8VCQr-mkozaKjbLjg   1   1          0            0       227b           227b         227b
yellow open   indice_estrategia_d_jerarquico Fw3jz5RWSg2OQrlrvWJ8nw   1   1          0            0       227b           227b         227b
yellow open   indice_estrategia_c_semantica  QGrNk11_RA-p7GDCoWFP9Q   1   1          0            0       227b           227b         227b
yellow open   indice_estrategia_b_markdown   TZHDlG80TMeIWpykBp2Emw   1   1          0            0       227b           227b         227b

